# Evaluierung & Benchmark: Faktenkonsistenz und Halluzinationserkennung

Dieses Notebook führt eine empirische Vergleichsstudie von **5 Bewertungsansätzen** zur Erkennung von **Faktenverzerrungen, Zahlenfehlern und semantischen Halluzinationen** durch:
1. **SBERT Baseline (Bi-Encoder):** 
2. **NLI Cross-Encoder:**  (Entailment / Contradiction)
3. **Klassischer NER-Overlap:** SpaCy  (Jaccard-Ähnlichkeit & Recall)
4. **Numerischer Slot-Check:** Reguläre Ausdrücke zur Erkennung nicht geerdeter Zahlen
5. **Hybride Multiplikative Metrik:** {\text{fact}} = R_{\text{SBERT}} \cdot (1 - P_{\text{contra}}) \cdot R_{\text{Num}}$

### Untersuchte 4 Testklassen ( = 160$ feste Textpaare aus ):
- **Klasse 1: Gold Positives (=50$):** Menschliche Referenzen (25 Lebenshilfe + 25 Master Parallelkorpus)
- **Klasse 2: Reale Modell-Halluzinationen (=50$):** Echte SFT/DPO-Ausgaben aus  (Deutschlandticket, Brand am Hasselbachplatz, Helga Schubert etc.)
- **Klasse 3: Random Shuffle Negatives (=30$):** Feste deterministisch versetzte Paare (vollständiger Themenwechsel)
- **Klasse 4: Kontrollierte Minimal-Perturbationen (=30$):** Gezielte synthetische Mutationen von Zahlen, Negationen, Rollen und Maßstäben

In [ ]:
import os
import sys
import glob
import json
import re
import random
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score
from IPython.display import display
import torch

def find_repo_root():
    p = os.path.abspath(os.getcwd())
    while p != os.path.dirname(p):
        if os.path.exists(os.path.join(p, 'data')) and os.path.exists(os.path.join(p, 'results')):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.path.expanduser('~/Documents/Master Thesis'))

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print('Arbeitsverzeichnis:', os.getcwd())
print(f'Nutze Device: {DEVICE}')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150
})


## 1. Evaluationsdaten laden

Einlesen des festen 160-Paare Benchmark-Datensatzes.

In [ ]:
CSV_PATH = os.path.join(REPO_ROOT, 'results/evaluation/factual_consistency_metric_results.csv')

df_res = pd.read_csv(CSV_PATH)
print(f'Ergebnisse geladen: {len(df_res)} Textpaare aus {CSV_PATH}')
print('Verteilung der Testklassen:')
for cat, cnt in df_res['category'].value_counts().items():
    print(f'  - {cat}: {cnt}')

display(df_res.head(3))


## 2. Quantitative Vergleichstabelle nach Testklasse


In [ ]:
metrics_cols ={
"sbert_score":"SBERT Similarity (R_sem)",
"nli_factuality":"NLI Factuality Score",
"nli_p_contra":"NLI P(Contradiction)",
"ner_jaccard":"NER Jaccard Overlap",
"number_consistency":"Number Consistency Check",
"composite_factuality":"Composite Multiplicative Score"
}

df_summary =df_res .groupby ("category")[list (metrics_cols .keys ())].mean ().rename (columns =metrics_cols )
df_std =df_res .groupby ("category")[list (metrics_cols .keys ())].std ().rename (columns =metrics_cols )

formatted_rows =[]
for cat in df_summary .index :
    row ={"Testklasse":cat }
    for col in df_summary .columns :
        row [col ]=f"{df_summary .loc [cat ,col ]:.4f} ± {df_std .loc [cat ,col ]:.4f}"
    formatted_rows .append (row )

df_table =pd .DataFrame (formatted_rows )
try :
    display (df_table )
except Exception :
    print (df_table .to_string (index =False ))


## 3. Verteilungsanalyse (4-Panel Boxplots)

In [ ]:
cat_map = {
    "1_Gold_Positives": "Gold Positives",
    "2_Real_Model_Hallucinations": "Real Hallucinations",
    "3_Random_Shuffle_Negatives": "Random Shuffle",
    "4_Targeted_Minimal_Perturbations": "Minimal Perturbations"
}
df_res["category_display"] = df_res["category"].map(lambda c: cat_map.get(c, c))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
palette = {
    "Gold Positives": "#2ecc71",
    "Real Hallucinations": "#e74c3c",
    "Random Shuffle": "#3498db",
    "Minimal Perturbations": "#9b59b6"
}

# SBERT Similarity
sns.boxplot(ax=axes[0, 0], data=df_res, x="category_display", y="sbert_score", hue="category_display", palette=palette, legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":8})
axes[0, 0].set_title("(A) SBERT Similarity (R_sem) - Bi-Encoder Blindspot", fontsize=13, fontweight="bold", pad=10)
axes[0, 0].set_ylabel("SBERT Score [0, 1]", fontsize=11, fontweight="bold")
axes[0, 0].set_xlabel("")

# NLI P(Contradiction)
sns.boxplot(ax=axes[0, 1], data=df_res, x="category_display", y="nli_p_contra", hue="category_display", palette=palette, legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":8})
axes[0, 1].set_title("(B) NLI Contradiction Probability (P_contra)", fontsize=13, fontweight="bold", pad=10)
axes[0, 1].set_ylabel("P(Contradiction) [0, 1] (Höher = Widerspruch)", fontsize=11, fontweight="bold")
axes[0, 1].set_xlabel("")

# NER Jaccard Overlap
sns.boxplot(ax=axes[1, 0], data=df_res, x="category_display", y="ner_jaccard", hue="category_display", palette=palette, legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":8})
axes[1, 0].set_title("(C) NER Jaccard Overlap", fontsize=13, fontweight="bold", pad=10)
axes[1, 0].set_ylabel("NER Jaccard [0, 1]", fontsize=11, fontweight="bold")
axes[1, 0].set_xlabel("Testklasse", fontsize=11, fontweight="bold")

# Composite Multiplicative Score
sns.boxplot(ax=axes[1, 1], data=df_res, x="category_display", y="composite_factuality", hue="category_display", palette=palette, legend=False, showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black", "markersize":8})
axes[1, 1].set_title("(D) Hybrider Composite Score (R_sem * R_NLI * R_Num)", fontsize=13, fontweight="bold", pad=10)
axes[1, 1].set_ylabel("Score [0, 1]", fontsize=11, fontweight="bold")
axes[1, 1].set_xlabel("Testklasse", fontsize=11, fontweight="bold")

# Einheitliche Y-Skala von 0 bis 1 auf allen 4 Subplots
for ax in axes.flat:
    ax.set_ylim(0.0, 1.05)
    ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.tick_params(axis="x", labelsize=11)
    ax.tick_params(axis="y", labelsize=10)
    ax.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plot_out = "research/img/analysis/factuality_metrics_4panel_comparison.png"
os.makedirs(os.path.dirname(plot_out), exist_ok=True)
plt.savefig(plot_out, dpi=300, bbox_inches="tight")
plt.show()
print(f"Plot gespeichert unter: {plot_out}")


## 4. ROC-Kurven & Trennkraft-Analyse

In [ ]:
plt .figure (figsize =(10 ,7 ))
y_true =df_res ["is_factually_correct"].values 

eval_configs =[
("SBERT Baseline",df_res ["sbert_score"].values ,"#3498db"),
("NLI Factuality (P_entail - P_contra)",df_res ["nli_factuality"].values ,"#e67e22"),
("NLI Contradiction (Inverted)",1.0 -df_res ["nli_p_contra"].values ,"#e74c3c"),
("NER Jaccard Overlap",df_res ["ner_jaccard"].values ,"#95a5a6"),
("Number Consistency Check",df_res ["number_consistency"].values ,"#9b59b6"),
("Hybrider Composite Score",df_res ["composite_factuality"].values ,"#2ecc71"),
]

for name ,scores ,color in eval_configs :
    auc =roc_auc_score (y_true ,scores )
    fpr ,tpr ,_ =roc_curve (y_true ,scores )
    plt .plot (fpr ,tpr ,label =f"{name } (AUC = {auc :.3f})",color =color ,lw =2 )

plt .plot ([0 ,1 ],[0 ,1 ],"k--",lw =1.5 ,label ="Zufall (AUC = 0.500)")
plt .xlabel ("False Positive Rate (1 - Spezifität)",fontsize =11 )
plt .ylabel ("True Positive Rate (Sensitivität)",fontsize =11 )
plt .title ("ROC-Kurven: Trennkraft zur Erkennung von Faktenfehlern & Halluzinationen",fontsize =13 ,fontweight ="bold")
plt .legend (loc ="lower right",fontsize =10 )

roc_out ="research/img/analysis/factuality_metrics_roc_curves.png"
plt .savefig (roc_out ,dpi =300 ,bbox_inches ="tight")
plt .show ()
print (f"ROC Plot gespeichert unter: {roc_out }")


## 5. Qualitative Fallstudien (Modell-Halluzinationen & Perturbationen)

In [ ]:
case_studies =df_res [df_res ["category"].isin (["2_Real_Model_Hallucinations","4_Targeted_Minimal_Perturbations"])].head (6 )

print ("="*95 )
print ("DETAIL-INSPEKTION: WIE REAGIEREN DIE METRIKEN AUF KONKRETE FEHLER?")
print ("="*95 )

for idx ,row in case_studies .iterrows ():
    print (f"\n[FALL {idx +1 }] Subtyp: {row ['subtype']} | Klasse: {row ['category']}")
    print (f"AS-QUELLE: {row ['as_text'][:120 ]}...")
    print (f"LS-ZIEL:   {row ['ls_text'][:120 ]}...")
    print (f"-> SBERT Score:        {row ['sbert_score']:.3f} (Täuschend hoch? {'JA'if row ['sbert_score']>0.80 else 'NEIN'})")
    print (f"-> NLI P(Contra):      {row ['nli_p_contra']:.3f} (Widerspruch erkannt? {'JA'if row ['nli_p_contra']>0.35 else 'NEIN'})")
    print (f"-> NER Jaccard:        {row ['ner_jaccard']:.3f}")
    print (f"-> Number Consistency: {row ['number_consistency']:.3f}")
    print (f"-> Composite Score:    {row ['composite_factuality']:.3f}")
    print ("-"*95 )
